<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_35_drf_fastapi/note_lesson_35_drf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔌 Урок 35 — DRF: API до нотаток як рефакторинг

| Крок | Що робимо |
|---|---|
| 0 | запускаємо `crispy_notes_project` з API у ноутбуці |
| 1 | серіалізатори: вихід і вхід (вправи 1–3) |
| 2 | API через `APIClient`: права і власник (вправи 4–6) |
| 3 | IDOR і `PATCH` (вправи 7–8) |
| 4 | знайди помилку: `ModelViewSet` віддає чужі нотатки (вправа 9) |
| 5 | той самий API на FastAPI (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія, діфи рефакторингу й архітектура — у книзі курсу: [Урок 35](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/); поглиблено — глава Django-книги [REST API: Django REST Framework](https://nikoriakviktot.github.io/notes_chat_app/06_application_architecture/drf_rest_api_full/).

---
## 0. Проєкт у ноутбуці

Проєкт — `crispy_notes_project` з уроку 34 + DRF: `hello_app/api.py` (серіалізатори й `NoteViewSet` поверх `services`/`selectors`) і роутер у `hello_project/urls.py`. У Colab клітинка нижче завантажить лише папку уроку і встановить залежності з `requirements.txt` проєкту.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_35_drf_fastapi"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("crispy_notes_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import crispy_bootstrap5, debug_toolbar, drf_spectacular, fastapi, httpx, rest_framework  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crispy_notes_project/requirements.txt",
                    "fastapi", "httpx"], check=True)

sys.path.insert(0, os.getcwd())                        # fastapi_notes.py лежить у папці уроку
os.chdir("crispy_notes_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 33)
django.setup()

from types import SimpleNamespace

from django.contrib.auth.models import User
from django.test.utils import setup_test_environment
from rest_framework.test import APIClient

from hello_app import selectors, services
from hello_app.api import NoteInputSerializer, NoteOutputSerializer
from hello_app.models import Note, Notebook

setup_test_environment()                               # хост testserver для тестового клієнта
User.objects.filter(username__in=["olena", "bob"]).delete()
olena = User.objects.create_user("olena", password="pass-12345")
bob = User.objects.create_user("bob", password="pass-12345")
study = Notebook.objects.create(user=olena, title="Навчання")
bob_notebook = Notebook.objects.create(user=bob, title="Записник Боба")
drf_note = services.create_note(user=olena, title="Вивчити DRF", content="serializers, viewsets", priority=3, notebook=study)
services.create_note(user=olena, title="Купити квитки", priority=2)
bob_note = services.create_note(user=bob, title="Нотатка Боба")
print("нотаток: Олена", olena.notes.count(), "| Боб", bob.notes.count())

---
## 1. Серіалізатори

`NoteOutputSerializer(note).data` — що бачить клієнт. `NoteInputSerializer(data=...)` — що клієнт може надіслати: `is_valid()` → `validated_data` або `errors`, як у форми.

### Вправа 1

Серіалізуй нотатку `drf_note` вихідним серіалізатором; словник — у `data`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
data = NoteOutputSerializer(drf_note).data
# END SOLUTION

assert data["title"] == "Вивчити DRF" and data["priority"] == 3
assert data["priority_label"] == "🟠 Високий" and data["notebook"] == "Навчання"
assert "user" not in data, "власника клієнт не бачить"
print({key: value for key, value in data.items() if key != "updated_at"})
print("✅ Вправа 1 пройдена")

### Вправа 2

Перевір вхідні дані `{"title": "", "priority": 9}`: множину полів з помилками — у `bad_fields`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
bad = NoteInputSerializer(data={"title": "", "priority": 9})
bad.is_valid()
bad_fields = set(bad.errors)
# END SOLUTION

assert bad_fields == {"title", "priority"}
for field, errors in bad.errors.items():
    print(field, "→", errors[0])
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Що буде в `validated_data` для `{'title': 'План'}` — лише `title`?

<details>
<summary>Відповідь</summary>

Ні: серіалізатор підставить значення за замовчуванням — `content=''`, `priority=1`, `is_pinned=False`, `notebook=None`. Інакше буде з `partial=True` (для `PATCH`): там у `validated_data` лише надіслані поля.

</details>

### Вправа 3

Вхідний серіалізатор фільтрує записники за користувачем з `context["request"]` — як `NoteForm(user=...)` в уроці 34. Перевір дані `{"title": "Чужий записник", "notebook": bob_notebook.pk}` від імені Олени (`context={"request": SimpleNamespace(user=olena)}`). Результат `is_valid()` — у `ok`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
serializer = NoteInputSerializer(data={"title": "Чужий записник", "notebook": bob_notebook.pk},
                                 context={"request": SimpleNamespace(user=olena)})
ok = serializer.is_valid()
# END SOLUTION

assert ok is False and "notebook" in serializer.errors
own = NoteInputSerializer(data={"title": "Свій записник", "notebook": study.pk}, context={"request": SimpleNamespace(user=olena)})
assert own.is_valid() and own.validated_data["notebook"] == study
print(serializer.errors["notebook"][0])
print("✅ Вправа 3 пройдена")

---
## 2. API через `APIClient`

`APIClient` — тестовий клієнт DRF: `format="json"` для тіла, `force_authenticate(user)` — «увійти» без пароля (лише для тестів).

### Вправа 4

Надішли `GET /api/notes/` **без входу**; код відповіді — у `anon_code`.

In [ ]:
api = APIClient()
# YOUR CODE HERE
# BEGIN SOLUTION
anon_code = api.get("/api/notes/").status_code
# END SOLUTION

assert anon_code == 403, "IsAuthenticated; перший клас автентифікації — сесійний, тому 403, а не 401"
print("✅ Вправа 4 пройдена")

### Вправа 5

Увійди як Олена (`force_authenticate`) і збережи заголовки її нотаток зі списку в `titles`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
api.force_authenticate(olena)
titles = [note["title"] for note in api.get("/api/notes/").json()]
# END SOLUTION

assert titles == ["Вивчити DRF", "Купити квитки"], "лише свої; спершу вищий пріоритет"
print("✅ Вправа 5 пройдена")

### Вправа 6

Створи нотатку `{"title": "Здати проєкт", "priority": 4, "user": bob.pk}` — з «підкинутим» власником. Відповідь — у `created`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
created = api.post("/api/notes/", {"title": "Здати проєкт", "priority": 4, "user": bob.pk}, format="json")
# END SOLUTION

assert created.status_code == 201
note = Note.objects.get(pk=created.json()["id"])
assert note.user == olena, "власника задає сервер (request.user), поле user ігнорується"
print(created.json()["priority_label"], "| власник:", note.user)
print("✅ Вправа 6 пройдена")

---
## 3. IDOR і `PATCH`

### Вправа 7

Спробуй прочитати й видалити нотатку Боба від імені Олени. Коди — у `read_code` і `delete_code`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
read_code = api.get(f"/api/notes/{bob_note.pk}/").status_code
delete_code = api.delete(f"/api/notes/{bob_note.pk}/").status_code
# END SOLUTION

assert (read_code, delete_code) == (404, 404), "selectors.get_note_detail шукає серед нотаток користувача"
assert Note.objects.filter(pk=bob_note.pk).exists()
print("✅ Вправа 7 пройдена")

**🔮 Прогноз:** Чому чужа нотатка — `404`, а не `403 Forbidden`?

<details>
<summary>Відповідь</summary>

`403` підтвердив би, що нотатка з таким `id` існує. `404` не розкриває нічого — перебирати чужі `id` марно.

</details>

### Вправа 8

Зміни лише заголовок нотатки «Здати проєкт» через `PATCH`: «Здати проєкт до п'ятниці». Відповідь — у `patched`.

In [ ]:
import time

note = Note.objects.get(title="Здати проєкт")
before = note.updated_at
time.sleep(0.01)
# YOUR CODE HERE
# BEGIN SOLUTION
patched = api.patch(f"/api/notes/{note.pk}/", {"title": "Здати проєкт до п'ятниці"}, format="json")
# END SOLUTION

note.refresh_from_db()
assert patched.status_code == 200
assert (note.title, note.priority) == ("Здати проєкт до п'ятниці", 4), "PATCH не чіпає ненадіслані поля"
assert note.updated_at > before, "update_note зберігає й updated_at (виправлення уроку 35)"
print("✅ Вправа 8 пройдена")

---
## 4. Знайди помилку

Колега «спростив» API — один `ModelViewSet` замість серіалізаторів і services:

```python
class QuickNoteViewSet(viewsets.ModelViewSet):
    queryset = Note.objects.all()
    serializer_class = NoteOutputSerializer
    permission_classes = [permissions.IsAuthenticated]
```

Вхід є, права є — але Олена бачить **усі** нотатки, разом із нотатками Боба (IDOR). Перевір і виправ.

### Вправа 9

Допиши `SafeNoteViewSet.get_queryset()`: нотатки лише поточного користувача — через `selectors.get_user_notes(self.request.user)`.

In [ ]:
from rest_framework import permissions, viewsets
from rest_framework.test import APIRequestFactory, force_authenticate


class QuickNoteViewSet(viewsets.ModelViewSet):
    queryset = Note.objects.all()
    serializer_class = NoteOutputSerializer
    permission_classes = [permissions.IsAuthenticated]


class SafeNoteViewSet(QuickNoteViewSet):
    def get_queryset(self):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        return selectors.get_user_notes(self.request.user)
        # END SOLUTION


def titles_for(viewset, user):
    request = APIRequestFactory().get("/notes/")
    force_authenticate(request, user=user)
    return sorted(note["title"] for note in viewset.as_view({"get": "list"})(request).data)


leak = titles_for(QuickNoteViewSet, olena)
assert "Нотатка Боба" in leak, "помилка відтворена: чужа нотатка у списку"
assert titles_for(SafeNoteViewSet, olena) == ["Вивчити DRF", "Здати проєкт до п'ятниці", "Купити квитки"]
assert titles_for(SafeNoteViewSet, bob) == ["Нотатка Боба"]
print("було:", leak)
print("✅ Вправа 9 пройдена")

---
## 5. Той самий API на FastAPI

`fastapi_notes.py` у папці уроку — ті самі поля й адреси, дані в пам'яті. Валідація — Pydantic, помилки — `422` зі списком `detail`.

### Вправа 10

Надішли в FastAPI-версію `{"title": "", "priority": 9}`. Код — у `fast_code`, назви полів з помилками (останній елемент `loc`) — у множину `fast_fields`.

In [ ]:
from fastapi.testclient import TestClient

from fastapi_notes import app

fast = TestClient(app)
# YOUR CODE HERE
# BEGIN SOLUTION
bad = fast.post("/api/notes/", json={"title": "", "priority": 9})
fast_code = bad.status_code
fast_fields = {error["loc"][-1] for error in bad.json()["detail"]}
# END SOLUTION

assert fast_code == 422 and fast_fields == {"title", "priority"}
drf_bad = api.post("/api/notes/", {"title": "", "priority": 9}, format="json")
print("DRF:", drf_bad.status_code, sorted(drf_bad.json()), "| FastAPI:", fast_code, sorted(fast_fields))
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Навіщо два серіалізатори — вхідний і вихідний?
2. Що станеться з полем `user` у тілі `POST`?
3. Чому `ViewSet`, а не `ModelViewSet`, у проєкті уроку?
4. Чим `PATCH` з `partial=True` відрізняється від `POST` щодо значень за замовчуванням?
5. Де DRF повертає `400`, а FastAPI — `422`? Що спільного?

<details>
<summary>Відповіді</summary>

1. Клієнт бачить більше, ніж може змінити (`id`, `priority_label`, `updated_at`); вхідний приймає лише дозволене.
2. Його немає у вхідному серіалізаторі — ігнорується; власника задає `request.user`.
3. `ModelViewSet` сам робить `Note.objects.all()` і `save()` — обійшов би `services`/`selectors` і (як у вправі 9) легко віддав би чужі дані.
4. `partial=True` не підставляє значень за замовчуванням: у `validated_data` лише надіслані поля — решта не змінюється.
5. Обидва — на помилки валідації вхідних даних з поясненням за полями; різний лише код і формат тіла.

</details>

## Далі

- Книга курсу: [Урок 35](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_35/) — рефакторинг «+ API», діфи, архітектура, Browsable API, «Знайди помилку» з `updated_at`.
- Запусти проєкт: `python manage.py runserver` у папці `crispy_notes_project`, увійди на сайт і відкрий `http://127.0.0.1:8000/api/notes/`.
- **Урок 36** — типізація й Pydantic: перший крок новинного агрегатора.